# 问题二：Raw 模型与上一时刻值的线性组合

本 notebook 聚焦于 `FILT. NTU` 原始值预测：

1. 使用 MLP、Random Forest、LightGBM 直接拟合当前 `FILT. NTU`；
2. 分析 Raw 模型的变量重要性；
3. 将三种模型的预测分别与上一时刻 `FILT. NTU` 做线性组合；
4. 在测试集上选择效果最好的组合，给出组合参数和对应模型的变量重要性。

线性组合参数仅使用训练集的 5 折 OOF（out-of-fold）预测估计，测试集只用于最终评估。

## 1. 导入依赖与参数

In [ ]:
from pathlib import Path
import warnings
import joblib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.base import clone
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import permutation_importance
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import KFold, train_test_split
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from lightgbm import LGBMRegressor

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 240)

TARGET_COL = "FILT. NTU"
RANDOM_STATE = 42
TEST_SIZE = 0.20
CORE_FEATURES = ["R/W NTU", "R/W PH", "AL_DOSE", "FRIDE_ABNORMAL", "R/W FLOW"]
EXOG_LEVEL_LAGS = list(range(0, 7))
TARGET_HISTORY_LAGS = [1, 2, 3]
MODEL_ORDER = ["MLP", "Random Forest", "LightGBM"]
MODEL_COLORS = {
    "MLP": "#4C78A8",
    "Random Forest": "#F58518",
    "LightGBM": "#54A24B",
}


## 2. 读取和预处理数据

In [ ]:
def locate_merged_file(filename="merged.xlsx"):
    cwd = Path.cwd().resolve()
    for root in [cwd] + list(cwd.parents):
        for candidate in [root / "data" / filename, root / filename]:
            if candidate.exists():
                return candidate
    matches = list(cwd.rglob(filename))
    if matches:
        return matches[0]
    raise FileNotFoundError(filename)


def construct_datetime(frame):
    frame = frame.copy()
    if "DATETIME" in frame.columns:
        frame["DATETIME"] = pd.to_datetime(frame["DATETIME"], errors="coerce")
        return frame
    date_col = next(c for c in ["DATE", "Date", "date"] if c in frame.columns)
    time_col = next(c for c in ["TIME", "Time", "time"] if c in frame.columns)
    frame["DATETIME"] = pd.to_datetime(
        frame[date_col].astype(str).str.split().str[0]
        + " "
        + frame[time_col].astype(str).str.split().str[-1],
        errors="coerce",
    )
    return frame


DATA_PATH = locate_merged_file()
PROJECT_ROOT = (
    DATA_PATH.parent.parent
    if DATA_PATH.parent.name == "data"
    else Path.cwd().resolve()
)
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "problem2_raw_linear_blend"
FIG_DIR = OUTPUT_DIR / "figures"
MODEL_DIR = OUTPUT_DIR / "models"
for directory in [OUTPUT_DIR, FIG_DIR, MODEL_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

df = construct_datetime(pd.read_excel(DATA_PATH))
df = df.dropna(subset=["DATETIME"]).sort_values("DATETIME").reset_index(drop=True)
input_features = [
    column
    for column in ["R/W NTU", "R/W PH", "ALUM", "F/RIDE", "R/W FLOW"]
    if column in df.columns
]
data = df[["DATETIME", TARGET_COL] + input_features].copy()

for column in [TARGET_COL] + input_features:
    data[column] = pd.to_numeric(data[column], errors="coerce")

for column, value in {"F/RIDE": 0, "R/W PH": 7, "ALUM": 0.06}.items():
    if column in data.columns:
        data[column] = data[column].fillna(value)

for column in [TARGET_COL] + input_features:
    if column not in {"F/RIDE", "R/W PH", "ALUM"}:
        data[column] = (
            data[column]
            .interpolate(method="linear", limit_direction="both")
            .ffill()
            .bfill()
        )

if not {"ALUM", "F/RIDE"}.issubset(data.columns):
    raise KeyError("需要同时存在 ALUM 和 F/RIDE 才能构造 AL_DOSE 与 FRIDE_ABNORMAL")

fride = data["F/RIDE"]
data["FRIDE_CLEAN"] = np.where((fride > 0) & (fride <= 0.01), fride, 0.0)
data["AL_DOSE"] = 56.9 * data["ALUM"] + 81.0 * data["FRIDE_CLEAN"]
data["FRIDE_ABNORMAL"] = (fride > 0.01).astype(int)

available_features = [column for column in CORE_FEATURES if column in data.columns]
data = data[["DATETIME", TARGET_COL] + available_features].copy()

data["PREVIOUS_FILT_NTU"] = data[TARGET_COL].shift(1)
data["DIFF_FILT_NTU"] = data[TARGET_COL] - data["PREVIOUS_FILT_NTU"]
data["DIFF_FILT_NTU_HURDLE01"] = data["DIFF_FILT_NTU"].clip(-1, 1).mask(
    data["DIFF_FILT_NTU"].clip(-1, 1).abs() < 0.1,
    0.0,
)
print("数据文件：", DATA_PATH)
print("数据规模：", data.shape)
display(data.head())


## 3. 构造 Raw 特征并选择 lag

In [ ]:
def safe_name(column):
    return (
        str(column)
        .replace("/", "_")
        .replace(" ", "_")
        .replace(".", "")
        .replace("+", "plus")
        .replace("-", "_")
    )


model_data = data.copy()
candidate_groups = {}
feature_display_names = {}

for variable in available_features:
    candidates = []
    for lag in EXOG_LEVEL_LAGS:
        feature = f"{safe_name(variable)}_lag{lag}"
        model_data[feature] = model_data[variable].shift(lag)
        candidates.append(feature)
        feature_display_names[feature] = f"{variable} (lag {lag})"
    candidate_groups[variable] = candidates

target_candidates = []
for lag in TARGET_HISTORY_LAGS:
    feature = f"FILT_NTU_lag{lag}"
    model_data[feature] = model_data[TARGET_COL].shift(lag)
    target_candidates.append(feature)
    feature_display_names[feature] = f"FILT. NTU (lag {lag})"
candidate_groups["FILT. NTU history"] = target_candidates

model_data = model_data.dropna(
    subset=["DATETIME", "PREVIOUS_FILT_NTU", TARGET_COL]
).reset_index(drop=True)
all_indices = np.arange(len(model_data))
hurdle_occurrence = (
    model_data["DIFF_FILT_NTU_HURDLE01"] != 0
).astype(int)
train_indices, test_indices = train_test_split(
    all_indices,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    shuffle=True,
    stratify=hurdle_occurrence,
)
train_indices = np.sort(train_indices)
test_indices = np.sort(test_indices)
train_subset = model_data.iloc[train_indices]

selection_records = []
selected_features = []
for group, candidates in candidate_groups.items():
    candidate_records = []
    for feature in candidates:
        pair = train_subset[[TARGET_COL, feature]].dropna()
        correlation = pair[feature].corr(pair[TARGET_COL], method="spearman")
        candidate_records.append(
            {
                "group": group,
                "feature": feature,
                "display_name": feature_display_names[feature],
                "spearman": correlation,
                "abs_spearman": abs(correlation),
                "n": len(pair),
            }
        )
    best = max(candidate_records, key=lambda record: record["abs_spearman"])
    selection_records.append(best)
    selected_features.append(best["feature"])

feature_selection_df = pd.DataFrame(selection_records)
display(feature_selection_df)
print("训练样本：", len(train_indices))
print("测试样本：", len(test_indices))
print("最终 Raw 特征：", selected_features)


## 4. 拟合三种 Raw 模型

In [ ]:
def build_regressors():
    return {
        "MLP": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", MLPRegressor(
                hidden_layer_sizes=(32, 16),
                activation="relu",
                solver="adam",
                alpha=1e-4,
                learning_rate_init=0.001,
                max_iter=2000,
                early_stopping=True,
                validation_fraction=0.15,
                random_state=RANDOM_STATE,
            )),
        ]),
        "Random Forest": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", RandomForestRegressor(
                n_estimators=500,
                random_state=RANDOM_STATE,
                max_features="sqrt",
                min_samples_leaf=2,
                n_jobs=-1,
            )),
        ]),
        "LightGBM": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", LGBMRegressor(
                n_estimators=500,
                learning_rate=0.03,
                num_leaves=31,
                subsample=0.85,
                colsample_bytree=0.85,
                random_state=RANDOM_STATE,
                n_jobs=-1,
                verbose=-1,
            )),
        ]),
    }


def regression_metrics(actual, predicted):
    return {
        "MAE": mean_absolute_error(actual, predicted),
        "RMSE": np.sqrt(mean_squared_error(actual, predicted)),
        "R2": r2_score(actual, predicted),
    }


X_train = model_data.iloc[train_indices][selected_features]
X_test = model_data.iloc[test_indices][selected_features]
y_train = model_data.iloc[train_indices][TARGET_COL]
y_test = model_data.iloc[test_indices][TARGET_COL]
previous_train = model_data.iloc[train_indices]["PREVIOUS_FILT_NTU"]
previous_test = model_data.iloc[test_indices]["PREVIOUS_FILT_NTU"]

raw_models = {}
raw_test_predictions = {}
raw_records = []

for model_name, model in build_regressors().items():
    model.fit(X_train, y_train)
    prediction = model.predict(X_test)
    raw_models[model_name] = model
    raw_test_predictions[model_name] = prediction
    raw_records.append(
        {
            "model": model_name,
            **regression_metrics(y_test, prediction),
        }
    )

raw_results_df = pd.DataFrame(raw_records)
persistence_metrics = regression_metrics(y_test, previous_test)
display(raw_results_df.sort_values("R2", ascending=False))
display(pd.DataFrame([{"model": "Previous FILT. NTU", **persistence_metrics}]))

plt.figure(figsize=(8, 5))
ordered_raw = raw_results_df.set_index("model").loc[MODEL_ORDER].reset_index()
plt.bar(
    ordered_raw["model"],
    ordered_raw["R2"],
    color=[MODEL_COLORS[name] for name in ordered_raw["model"]],
)
plt.axhline(
    persistence_metrics["R2"],
    color="black",
    linestyle="--",
    label="Previous FILT. NTU",
)
plt.ylabel("Test R²")
plt.title("Raw Model Performance")
plt.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / "raw_model_r2.png", dpi=300, bbox_inches="tight")
plt.show()


## 5. Raw 模型的解释性问题

Raw 模型的测试结果看起来较好，但变量重要性通常会显示：**上一时刻的 `FILT. NTU` 是主要预测来源**。由于水质序列本身具有较强的时间连续性，模型很容易通过复制或修正上一时刻值获得较高的 R²。

因此，这类模型虽然预测精度较高，但解释性有限：原水指标（如 `R/W NTU`、`R/W PH`）和操作变量（如 `ALUM`、`F/RIDE`、`R/W FLOW`）的独立影响容易被上一时刻目标值掩盖。下面使用统一的 permutation importance 量化这一现象。

In [ ]:
importance_records = []
for model_name, model in raw_models.items():
    result = permutation_importance(
        model,
        X_test,
        y_test,
        scoring="r2",
        n_repeats=20,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )
    for feature, mean_value, std_value in zip(
        selected_features,
        result.importances_mean,
        result.importances_std,
    ):
        importance_records.append(
            {
                "model": model_name,
                "feature": feature,
                "variable": feature_display_names[feature],
                "importance_R2_drop": mean_value,
                "importance_std": std_value,
            }
        )

importance_df = pd.DataFrame(importance_records)
importance_df["positive_importance"] = importance_df[
    "importance_R2_drop"
].clip(lower=0)
importance_df["relative_importance"] = importance_df.groupby("model")[
    "positive_importance"
].transform(lambda values: values / values.sum() if values.sum() else 0)

display(
    importance_df.sort_values(
        ["model", "importance_R2_drop"],
        ascending=[True, False],
    )
)

fig, axes = plt.subplots(1, 3, figsize=(17, 5), sharey=True)
for axis, model_name in zip(axes, MODEL_ORDER):
    plot_data = (
        importance_df.loc[importance_df["model"].eq(model_name)]
        .sort_values("importance_R2_drop")
    )
    axis.barh(
        plot_data["variable"],
        plot_data["importance_R2_drop"],
        color=MODEL_COLORS[model_name],
    )
    axis.axvline(0, color="black", linewidth=0.8)
    axis.set_title(model_name)
    axis.set_xlabel("Permutation importance (R² drop)")
fig.suptitle("Raw Model Feature Importance")
plt.tight_layout()
plt.savefig(FIG_DIR / "raw_model_feature_importance.png", dpi=300, bbox_inches="tight")
plt.show()


## 6. 模型预测与上一时刻值的线性组合

对每一种 Raw 模型建立：

```text
组合预测 = β_model × Raw模型预测 + β_previous × 上一时刻FILT.NTU + intercept
```

组合系数使用训练集 5 折 OOF 预测拟合。随后用全部训练数据重新训练 Raw 模型，并在测试集上计算组合预测，测试数据不参与系数估计。

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
blend_records = []
blend_models = {}
blend_test_predictions = {}
oof_predictions = {}

for model_name, model_template in build_regressors().items():
    oof_prediction = np.zeros(len(X_train))
    for fold_train, fold_valid in kf.split(X_train):
        fold_model = clone(model_template)
        fold_model.fit(X_train.iloc[fold_train], y_train.iloc[fold_train])
        oof_prediction[fold_valid] = fold_model.predict(X_train.iloc[fold_valid])

    oof_predictions[model_name] = oof_prediction
    blend_train_inputs = pd.DataFrame(
        {
            "model_prediction": oof_prediction,
            "previous_FILT_NTU": previous_train.to_numpy(),
        }
    )
    combiner = Pipeline([
        ("scaler", StandardScaler()),
        ("model", LinearRegression()),
    ])
    combiner.fit(blend_train_inputs, y_train)

    raw_prediction = raw_test_predictions[model_name]
    blend_test_inputs = pd.DataFrame(
        {
            "model_prediction": raw_prediction,
            "previous_FILT_NTU": previous_test.to_numpy(),
        }
    )
    blend_prediction = combiner.predict(blend_test_inputs)

    blend_scaler = combiner.named_steps["scaler"]
    blend_linear_model = combiner.named_steps["model"]
    standardized_beta = blend_linear_model.coef_
    original_beta = standardized_beta / blend_scaler.scale_
    original_intercept = (
        blend_linear_model.intercept_
        - np.sum(standardized_beta * blend_scaler.mean_ / blend_scaler.scale_)
    )
    blend_models[model_name] = combiner
    blend_test_predictions[model_name] = blend_prediction
    blend_records.append(
        {
            "model": model_name,
            "standardized_beta_model": standardized_beta[0],
            "standardized_beta_previous": standardized_beta[1],
            "beta_model": original_beta[0],
            "beta_previous": original_beta[1],
            "intercept": original_intercept,
            **{f"raw_{key}": value for key, value in regression_metrics(y_test, raw_prediction).items()},
            **{f"blend_{key}": value for key, value in regression_metrics(y_test, blend_prediction).items()},
        }
    )

blend_results_df = pd.DataFrame(blend_records)
blend_results_df["R2_gain"] = (
    blend_results_df["blend_R2"] - blend_results_df["raw_R2"]
)
blend_results_df["RMSE_reduction"] = (
    blend_results_df["raw_RMSE"] - blend_results_df["blend_RMSE"]
)
blend_results_df = blend_results_df.sort_values(
    ["blend_R2", "blend_RMSE"],
    ascending=[False, True],
).reset_index(drop=True)
display(blend_results_df)

plot_blend = blend_results_df.set_index("model").loc[MODEL_ORDER].reset_index()
x = np.arange(len(plot_blend))
width = 0.36
plt.figure(figsize=(9, 5))
plt.bar(x - width / 2, plot_blend["raw_R2"], width, label="Raw")
plt.bar(x + width / 2, plot_blend["blend_R2"], width, label="Linear blend")
plt.axhline(
    persistence_metrics["R2"],
    color="black",
    linestyle="--",
    label="Previous FILT. NTU",
)
plt.xticks(x, plot_blend["model"])
plt.ylabel("Test R²")
plt.title("Raw Models vs Linear Blends")
plt.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / "raw_vs_linear_blend_r2.png", dpi=300, bbox_inches="tight")
plt.show()


## 7. 最佳组合、参数和变量重要性

In [ ]:
best_blend = blend_results_df.iloc[0]
best_model_name = best_blend["model"]
best_model_importance_df = (
    importance_df.loc[importance_df["model"].eq(best_model_name)]
    .sort_values("importance_R2_drop", ascending=False)
    .reset_index(drop=True)
)

print("最佳线性组合基础模型：", best_model_name)
print(f"组合测试 R²：{best_blend['blend_R2']:.6f}")
print(f"组合测试 MAE：{best_blend['blend_MAE']:.6f}")
print(f"组合测试 RMSE：{best_blend['blend_RMSE']:.6f}")
print(
    "线性组合公式："
    f" {best_blend['beta_model']:.6f} × {best_model_name}预测"
    f" + {best_blend['beta_previous']:.6f} × 上一时刻FILT.NTU"
    f" + {best_blend['intercept']:.6f}"
)
display(best_model_importance_df)

best_prediction_df = pd.DataFrame(
    {
        "DATETIME": model_data.iloc[test_indices]["DATETIME"].to_numpy(),
        "actual_FILT_NTU": y_test.to_numpy(),
        "previous_FILT_NTU": previous_test.to_numpy(),
        "raw_prediction": raw_test_predictions[best_model_name],
        "blend_prediction": blend_test_predictions[best_model_name],
    }
)

plt.figure(figsize=(9, 5))
plot_importance = best_model_importance_df.sort_values("importance_R2_drop")
plt.barh(
    plot_importance["variable"],
    plot_importance["importance_R2_drop"],
    color=MODEL_COLORS[best_model_name],
)
plt.axvline(0, color="black", linewidth=0.8)
plt.xlabel("Permutation importance (R² drop)")
plt.title(f"Feature Importance of Best Blend Base Model: {best_model_name}")
plt.tight_layout()
plt.savefig(FIG_DIR / "best_blend_model_feature_importance.png", dpi=300, bbox_inches="tight")
plt.show()


## 8. 结论

- Raw 模型可以获得较高的原始值预测 R²；
- 但模型的重要性主要集中于上一时刻 `FILT. NTU`，说明其优势很大程度来自时间连续性；
- 因而仅依靠 Raw 模型较难清晰解释原水指标与操作变量对出水浊度变化的作用；
- 将 Raw 模型预测与上一时刻值分开进行线性组合，可以显式量化两部分贡献，并在不使用测试集调参的前提下比较三种组合。

## 9. 保存结果

In [ ]:
results_path = OUTPUT_DIR / "problem2_raw_linear_blend_results.xlsx"
predictions_path = OUTPUT_DIR / "best_linear_blend_predictions.xlsx"

with pd.ExcelWriter(results_path, engine="openpyxl") as writer:
    feature_selection_df.to_excel(writer, sheet_name="selected_features", index=False)
    raw_results_df.to_excel(writer, sheet_name="raw_results", index=False)
    blend_results_df.to_excel(writer, sheet_name="blend_results", index=False)
    importance_df.to_excel(writer, sheet_name="all_feature_importance", index=False)
    best_model_importance_df.to_excel(
        writer,
        sheet_name="best_model_importance",
        index=False,
    )
    best_prediction_df.to_excel(writer, sheet_name="best_predictions", index=False)

best_prediction_df.to_excel(predictions_path, index=False)

for model_name, model in raw_models.items():
    joblib.dump(
        {
            "raw_model": model,
            "linear_combiner": blend_models[model_name],
            "features": selected_features,
        },
        MODEL_DIR / f"{model_name.replace(' ', '_')}_raw_linear_blend.joblib",
    )

print("结果文件：", results_path)
print("最佳组合预测：", predictions_path)
